# Confounder analysis — Table S2

Path2Space predicts spatial gene expression from H&E histology images. This
notebook checks whether those predictions reflect **genuine biological**
**variation** rather than technical or compositional artifacts — by measuring
how much of the agreement between predicted and measured expression survives
once a potential confounder is controlled for.

Three confounders are tested, each correlated with expression for a different
reason:

| Confounder | Why it could be a confounder | How it is measured |
|---|---|---|
| Hematoxylin stain intensity | Directly visible in the input H&E image — a model could key on staining rather than biology | mean hematoxylin optical density per spot (see the `qc` component) |
| Cancer cell fraction | Tumor-dense regions have distinct expression; a model could merely track tumor density | SpaCET deconvolution of the measured spatial transcriptomics |
| Total RNA content per spot | Spots with more RNA show higher counts for most genes | log1p of total UMI counts per spot |

For every gene, in every slide, we compute the **partial correlation** between
predicted and measured expression while controlling for the confounder. If that
partial correlation stays positive and significant, the prediction carries
gene-specific spatial information *beyond* the confounder.

## Method

The analysis runs in two stages; the functions live in
[`lib/confounder_stats.py`](../lib/confounder_stats.py).

**Stage A — per slide, per gene.**
For each slide and gene: Pearson correlations of predicted and measured
expression with the confounder, and the partial correlation between predicted
and measured expression controlling for the confounder, computed by
residual-based regression (`partial_correlation` — standardize the three
variables, regress predicted and measured expression on the confounder,
correlate the residuals).

Stage A needs the full per-slide spatial-transcriptomics dataset — predicted
and measured expression matrices, SpaCET deconvolution, and per-spot QC — which
is large and is **not** bundled here. Its output *is* bundled:
`data/per_slide_correlations.parquet`, the per-slide / per-gene correlation
table for 40 slides and 14,068 genes.

**Stage B — aggregate across slides** (this notebook).
Per gene: spot-count-weighted mean correlations; per-slide p-values combined
with Stouffer's method (weights proportional to `sqrt(n_spots)`); and
Benjamini-Hochberg FDR across genes. Slides are aggregated three ways — all
cohorts, cross-validation only (held-out folds of the TNBC training cohort),
and external validation only (HEST, HTAN, pierre_martinez).

## Setup

In [1]:
import sys; sys.path.insert(0, "../lib")
import pandas as pd

from confounder_stats import aggregate_per_gene, confounder_summary

## Load the per-slide correlation table

The bundled Stage-A output. One row per (slide, gene): the predicted-vs-measured
correlation, each confounder's correlation with predicted and measured
expression, and the partial correlations — each with a p-value.

In [2]:
per_slide = pd.read_parquet("../data/per_slide_correlations.parquet")

print(f"{len(per_slide):,} rows  |  {per_slide['slide'].nunique()} slides  |  "
      f"{per_slide['gene'].nunique():,} genes")
print("cohorts:", per_slide['cohort'].value_counts().to_dict())
per_slide.head()

548,340 rows  |  40 slides  |  14,068 genes
cohorts: {'TNBC': 309496, 'HTAN': 117251, 'HEST': 65337, 'pierre_martinez': 56256}


,slide,patient,cohort,gene,n_spots,pct_spots_shared,corr_pred_vs_obs,corr_pred_vs_obs_pvalue,corr_pred_vs_tumor,corr_pred_vs_tumor_pvalue,...,corr_obs_vs_H,corr_obs_vs_H_pvalue,partial_corr_given_H,partial_corr_H_pvalue,corr_pred_vs_counts,corr_pred_vs_counts_pvalue,corr_obs_vs_counts,corr_obs_vs_counts_pvalue,partial_corr_given_counts,partial_corr_counts_pvalue
0,GSM6592058_M11_CL_like3,GSM6592058_M11_CL_like3,pierre_martinez,TOLLIP-AS1,1377,98.007118,0.213358,1.220244e-15,0.513922,1.128169e-93,...,0.310460,3.769596e-32,0.130214,1.256624e-06,0.521867,4.874449e-97,0.405732,1.044201e-55,0.002077,0.938636
1,GSM6592058_M11_CL_like3,GSM6592058_M11_CL_like3,pierre_martinez,ZNF155,1377,98.007118,0.321057,2.200217e-34,0.374545,4.229600e-47,...,0.299465,6.300065e-30,0.160638,2.068890e-09,0.603032,3.924998e-137,0.522700,2.137937e-97,0.008604,0.749808
2,GSM6592058_M11_CL_like3,GSM6592058_M11_CL_like3,pierre_martinez,RSKR,1377,98.007118,0.214425,8.739013e-16,0.377691,6.304354e-48,...,0.158771,3.143552e-09,0.164019,9.348700e-10,0.493166,2.712174e-85,0.374268,4.997537e-47,0.037001,0.170139
3,GSM6592058_M11_CL_like3,GSM6592058_M11_CL_like3,pierre_martinez,MED18,1377,98.007118,0.137517,3.011862e-07,0.241907,8.699584e-20,...,0.471376,4.284554e-77,0.053244,4.830511e-02,0.248402,8.253346e-21,0.606959,2.255708e-139,-0.017215,0.523432
4,GSM6592058_M11_CL_like3,GSM6592058_M11_CL_like3,pierre_martinez,GON7,1377,98.007118,0.490685,2.494287e-84,0.550946,3.813287e-110,...,0.403349,5.123755e-55,0.327658,0.000000e+00,0.679219,6.021239e-187,0.615639,1.939582e-144,0.125407,0.000003


## Stage B — aggregate per gene

`aggregate_per_gene` collapses the per-slide rows to one row per gene:
spot-count-weighted mean correlations, Stouffer-combined p-values, and a
Benjamini-Hochberg `*_fdr` column for each p-value. We aggregate all slides,
then the cross-validation and external-validation subsets separately.

In [3]:
CV_COHORTS = ["TNBC"]   # cross-validation: held-out folds of the training cohort
is_cv = per_slide["cohort"].isin(CV_COHORTS)

gene_all = aggregate_per_gene(per_slide)
gene_cv  = aggregate_per_gene(per_slide[is_cv])
gene_ext = aggregate_per_gene(per_slide[~is_cv])

print(f"per-gene table: {gene_all.shape[0]:,} genes x {gene_all.shape[1]} columns")
gene_all.head()

per-gene table: 14,068 genes x 33 columns


,gene,n_slides,total_spots,corr_pred_vs_obs,corr_pred_vs_obs_pvalue,corr_pred_vs_tumor,corr_pred_vs_tumor_pvalue,corr_obs_vs_tumor,corr_obs_vs_tumor_pvalue,partial_corr_given_tumor,...,corr_pred_vs_obs_fdr,corr_pred_vs_tumor_fdr,corr_obs_vs_tumor_fdr,partial_corr_tumor_fdr,corr_pred_vs_H_fdr,corr_obs_vs_H_fdr,partial_corr_H_fdr,corr_pred_vs_counts_fdr,corr_obs_vs_counts_fdr,partial_corr_counts_fdr
0,A1BG,39,76654,0.246002,0.000000e+00,0.375105,0.0,0.523922,0.000000e+00,0.093500,...,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.0,0.0,0.0,0.000000e+00
1,A1BG-AS1,34,62029,0.170094,0.000000e+00,0.413671,0.0,0.333058,0.000000e+00,0.046538,...,0.000000e+00,0.0,0.000000e+00,8.963757e-280,0.0,0.000000e+00,0.0,0.0,0.0,1.869116e-136
2,A2M,40,80949,0.212925,0.000000e+00,0.277863,0.0,0.397017,0.000000e+00,0.130397,...,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.0,0.0,0.0,0.000000e+00
3,A2M-AS1,34,62029,0.088031,1.021218e-146,0.376370,0.0,0.147322,2.337615e-306,0.036507,...,1.072611e-146,0.0,2.395166e-306,4.131652e-69,0.0,2.670944e-109,0.0,0.0,0.0,5.237994e-50
4,A4GALT,40,80949,0.116431,0.000000e+00,0.252463,0.0,0.366945,0.000000e+00,0.043272,...,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.0,0.0,0.0,0.000000e+00


## Output table 1 — confounder summary

One row per confounder: how strongly predicted and measured expression
correlate with it, and how many genes keep a significant positive partial
correlation once it is controlled for (FDR < 0.05).

In [4]:
summary = confounder_summary(gene_all, gene_cv, gene_ext)
summary

,Confounder,Correlation with measured expression,Correlation with predicted expression,Genes with significant positive partial correlation (n),Genes with significant positive partial correlation (%),Median partial correlation (all cohorts),Median partial correlation (cross-validation),Median partial correlation (external validation)
0,Hematoxylin intensity,0.22,0.30,14021,99.7,0.30,0.34,0.27
1,Total RNA content,0.54,0.44,13788,98.0,0.13,0.13,0.14
2,Cancer cell fraction,0.52,0.47,13897,98.8,0.11,0.12,0.11


In [5]:
# A plain-language readout, generated from the table above.
n_genes = len(gene_all)
for _, r in summary.iterrows():
    print(f"{r['Confounder']}:")
    print(f"  measured / predicted expression correlate with it at "
          f"r = {r['Correlation with measured expression']:.2f} / "
          f"{r['Correlation with predicted expression']:.2f}")
    n  = r['Genes with significant positive partial correlation (n)']
    pc = r['Genes with significant positive partial correlation (%)']
    print(f"  yet {n:,}/{n_genes:,} genes ({pc}%) keep a significant positive "
          f"partial correlation (FDR < 0.05),")
    print(f"  median partial r = {r['Median partial correlation (all cohorts)']:.2f}\n")

Hematoxylin intensity:
  measured / predicted expression correlate with it at r = 0.22 / 0.30
  yet 14,021/14,068 genes (99.7%) keep a significant positive partial correlation (FDR < 0.05),
  median partial r = 0.30

Total RNA content:
  measured / predicted expression correlate with it at r = 0.54 / 0.44
  yet 13,788/14,068 genes (98.0%) keep a significant positive partial correlation (FDR < 0.05),
  median partial r = 0.13

Cancer cell fraction:
  measured / predicted expression correlate with it at r = 0.52 / 0.47
  yet 13,897/14,068 genes (98.8%) keep a significant positive partial correlation (FDR < 0.05),
  median partial r = 0.11



## Output table 2 — gene-level statistics

The full per-gene table behind the summary: for every gene, the weighted-mean
correlations, Stouffer-combined p-values and BH-FDR for each confounder. This
is the gene-resolution view of Table S2.

In [6]:
# partial correlation + FDR for the three confounders, best genes first
view = ["gene", "n_slides", "total_spots",
        "partial_corr_given_H", "partial_corr_H_fdr",
        "partial_corr_given_tumor", "partial_corr_tumor_fdr",
        "partial_corr_given_counts", "partial_corr_counts_fdr"]
gene_all[view].sort_values("partial_corr_given_H", ascending=False).head(10)

,gene,n_slides,total_spots,partial_corr_given_H,partial_corr_H_fdr,partial_corr_given_tumor,partial_corr_tumor_fdr,partial_corr_given_counts,partial_corr_counts_fdr
4913,GAS5,32,54301,0.554547,0.0,0.339102,0.0,0.388071,0.0
7772,NAA20,40,80949,0.537522,0.0,0.264262,0.0,0.346833,0.0
641,ADIPOR1,40,80949,0.537436,0.0,0.273884,0.0,0.349553,0.0
2489,CD46,40,80949,0.536783,0.0,0.273741,0.0,0.355585,0.0
13331,WRNIP1,40,80949,0.536010,0.0,0.280293,0.0,0.354966,0.0
2949,CNBP,40,80949,0.535431,0.0,0.286180,0.0,0.363781,0.0
1427,ARPC1A,40,80949,0.534731,0.0,0.270259,0.0,0.363456,0.0
9330,PRELID3B,40,80949,0.532687,0.0,0.264869,0.0,0.347653,0.0
7424,MORF4L2,40,80949,0.532549,0.0,0.276854,0.0,0.353984,0.0
1600,ATP6V1G1,40,80949,0.532532,0.0,0.299613,0.0,0.368767,0.0


## Save the tables

Both tables are written as pickles — no Excel.

In [7]:
summary.to_pickle("../data/confounder_summary.pkl")
gene_all.to_pickle("../data/gene_level_confounder_stats.pkl")
print("wrote ../data/confounder_summary.pkl")
print("wrote ../data/gene_level_confounder_stats.pkl")

wrote ../data/confounder_summary.pkl
wrote ../data/gene_level_confounder_stats.pkl


## Takeaway

Each confounder genuinely correlates with both measured and predicted
expression — staining intensity, tumor density and RNA yield are all real
sources of variation. Yet after controlling for any one of them, the large
majority of genes retain a significant positive partial correlation between
predicted and measured expression. Path2Space predictions therefore capture
gene-specific spatial expression variation that is not explained by global
staining intensity, tumor density, or RNA content.